# CreditWise V4
## Explainable Loan Approval and Credit Risk Prediction

CreditWise predicts loan approval from applicant financial and demographic information. The notebook builds a leakage-safe machine-learning workflow, compares multiple classifiers, explains the final model with SHAP, and evaluates how probability thresholds affect lending decisions.

**Target:** `Loan_Approved`  
**Primary model-selection metric:** ROC-AUC  
**Final evaluation:** untouched 20% stratified test set


In [ ]:
# Run once only if required:
# %pip install numpy pandas matplotlib scikit-learn xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed.")
    print("Run: %pip install xgboost")

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)


In [ ]:
# V4 dependencies
# Run once if needed:
# %pip install shap joblib

import joblib
from pathlib import Path

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP is not installed. Run: %pip install shap")

from sklearn.model_selection import RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss
)


# 1. Dataset

The raw loan dataset is loaded and audited before modeling. Rows without a ground-truth target cannot be used for supervised learning, while missing predictor values are retained for pipeline-based imputation.


In [ ]:
DATA_PATH = "loan_approval_data.csv"

df_raw = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", df_raw.shape)
display(df_raw.head())


In [ ]:
print("Columns:")
print(df_raw.columns.tolist())

print("\nData types:")
display(df_raw.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = df_raw.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

print("\nExact duplicate rows:", df_raw.duplicated().sum())

print("\nTarget distribution including missing targets:")
print(df_raw["Loan_Approved"].value_counts(dropna=False))


In [ ]:
df = df_raw.dropna(subset=["Loan_Approved"]).copy()
df = df.drop_duplicates().reset_index(drop=True)

# Standardize whitespace in text columns.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(
        lambda value: value.strip() if isinstance(value, str) else value
    )

print("Labeled dataset shape:", df.shape)
print("\nTarget counts:")
print(df["Loan_Approved"].value_counts())

print("\nTarget percentages:")
print((df["Loan_Approved"].value_counts(normalize=True) * 100).round(2))


# 2. Data Quality and Leakage Controls

`Applicant_ID` and other identifier-like fields are excluded because identifiers should not carry predictive meaning. Columns whose names could directly reveal the outcome are also flagged and excluded.


In [ ]:
identifier_columns = [
    col for col in df.columns
    if (
        col.lower() in {"id", "applicant_id", "customer_id", "loan_id"}
        or col.lower().endswith("_id")
    )
]

target_like_columns = [
    col for col in df.columns
    if (
        col != "Loan_Approved"
        and any(
            token in col.lower()
            for token in ["approved", "approval", "decision", "target", "label", "outcome"]
        )
    )
]

print("Identifier columns excluded:", identifier_columns)
print("Potential target-like columns excluded:", target_like_columns)

excluded_predictors = list(
    dict.fromkeys(identifier_columns + target_like_columns)
)


# 3. Exploratory Analysis

The analysis summarizes feature distributions, class balance, and numeric correlations to understand the modeling data before training.


In [ ]:
display(df.describe(include="all").T)


In [ ]:
plt.figure(figsize=(6, 4))
df["Loan_Approved"].value_counts().plot(kind="bar")
plt.title("Loan Approval Class Distribution")
plt.xlabel("Loan Approved")
plt.ylabel("Number of Applicants")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
numeric_eda = df.select_dtypes(include=np.number)

if not numeric_eda.empty:
    correlation = numeric_eda.corr()

    plt.figure(figsize=(12, 8))
    plt.imshow(correlation, aspect="auto")
    plt.colorbar()
    plt.xticks(
        range(len(correlation.columns)),
        correlation.columns,
        rotation=90
    )
    plt.yticks(
        range(len(correlation.columns)),
        correlation.columns
    )
    plt.title("Numeric Feature Correlation Matrix")
    plt.tight_layout()
    plt.show()


# 4. Feature Engineering

A small set of domain-relevant features is derived from income, loan amount, savings, collateral, and dependents. Ratio calculations protect against zero denominators, and non-finite values are converted to missing values for later imputation.


In [ ]:
def safe_ratio(numerator, denominator, epsilon=1e-6):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")

    safe_denominator = denominator.where(
        denominator.abs() > epsilon,
        np.nan
    )

    ratio = numerator / safe_denominator
    return ratio.replace([np.inf, -np.inf], np.nan)


def add_credit_features(dataframe):
    data = dataframe.copy()

    # Log income reduces right-skew while keeping the raw income feature.
    if "Applicant_Income" in data.columns:
        applicant_income = pd.to_numeric(
            data["Applicant_Income"],
            errors="coerce"
        )
        data["Applicant_Income_log"] = np.log1p(
            applicant_income.clip(lower=0)
        )

    income_columns = [
        col
        for col in [
            "Applicant_Income",
            "Coapplicant_Income",
            "Co_Applicant_Income"
        ]
        if col in data.columns
    ]

    if income_columns:
        income_frame = data[income_columns].apply(
            pd.to_numeric,
            errors="coerce"
        )

        data["Total_Income"] = income_frame.sum(
            axis=1,
            min_count=1
        )

        if "Loan_Amount" in data.columns:
            data["Loan_to_Income"] = safe_ratio(
                data["Loan_Amount"],
                data["Total_Income"]
            )

        if (
            "Dependents" in data.columns
            and pd.api.types.is_numeric_dtype(data["Dependents"])
        ):
            data["Income_per_Dependent"] = safe_ratio(
                data["Total_Income"],
                data["Dependents"].fillna(0) + 1
            )

    if {"Savings", "Loan_Amount"}.issubset(data.columns):
        data["Savings_to_Loan"] = safe_ratio(
            data["Savings"],
            data["Loan_Amount"]
        )

    if {"Collateral_Value", "Loan_Amount"}.issubset(data.columns):
        data["Collateral_Coverage"] = safe_ratio(
            data["Collateral_Value"],
            data["Loan_Amount"]
        )

    # Defensive numerical cleanup.
    data = data.replace([np.inf, -np.inf], np.nan)

    return data


df_model = add_credit_features(df)

engineered_features = [
    col for col in df_model.columns
    if col not in df.columns
]

print("Engineered features:")
print(engineered_features)


# 5. Numerical Quality Check

Engineered numeric features are checked for missing values and extreme ranges before modeling.


In [ ]:
numeric_model = df_model.select_dtypes(include=np.number)

audit = []

for col in numeric_model.columns:
    values = (
        numeric_model[col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(values) > 0:
        audit.append({
            "Feature": col,
            "Min": values.min(),
            "Median": values.median(),
            "Max": values.max(),
            "Missing": numeric_model[col].isna().sum()
        })

range_audit = pd.DataFrame(audit)

display(range_audit)


# 6. Modeling Dataset

The target is encoded as `Rejected = 0` and `Approved = 1`. A stratified 80/20 split preserves the class distribution in training and test data.


In [ ]:
columns_to_drop = [
    "Loan_Approved",
    *excluded_predictors
]

X = df_model.drop(
    columns=[
        col for col in columns_to_drop
        if col in df_model.columns
    ]
).copy()

target_text = (
    df_model["Loan_Approved"]
    .astype(str)
    .str.strip()
    .str.lower()
)

target_map = {
    "no": 0,
    "yes": 1,
    "0": 0,
    "1": 1,
    "false": 0,
    "true": 1
}

y = target_text.map(target_map)

if y.isna().any():
    unexpected = df_model.loc[
        y.isna(),
        "Loan_Approved"
    ].unique()

    raise ValueError(
        f"Unexpected Loan_Approved labels: {unexpected}"
    )

y = y.astype(int)

print("Excluded predictors:", excluded_predictors)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining proportions:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting proportions:")
print(y_test.value_counts(normalize=True).round(4))


# 7. Leakage-Safe Preprocessing

Numeric variables use median imputation and standardization. Categorical variables use most-frequent imputation and one-hot encoding. These transformations remain inside the scikit-learn pipeline so they are fitted only on training data within each split or cross-validation fold.


In [ ]:
numeric_features = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore")
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


# 8. Baseline Model Comparison

Logistic Regression provides an interpretable linear baseline, while Random Forest and XGBoost capture nonlinear relationships. All models are evaluated using the same stratified 5-fold cross-validation procedure.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

if XGBOOST_AVAILABLE:
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()

    scale_pos_weight = (
        negative_count / positive_count
        if positive_count > 0
        else 1.0
    )

    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.90,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

print("Candidate models:")
for name in models:
    print("-", name)


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_rows = []

for model_name, estimator in models.items():

    pipeline = Pipeline([
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "model",
            clone(estimator)
        )
    ])

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise"
    )

    cv_rows.append({
        "Model": model_name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),
        "ROC-AUC": scores["test_roc_auc"].mean(),
        "ROC-AUC Std": scores["test_roc_auc"].std()
    })


cv_results = (
    pd.DataFrame(cv_rows)
    .sort_values(
        ["ROC-AUC", "F1"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(cv_results.round(4))


In [ ]:
test_rows = []
fitted_models = {}

for model_name, estimator in models.items():

    pipeline = Pipeline([
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "model",
            clone(estimator)
        )
    ])

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_test
    )

    probabilities = pipeline.predict_proba(
        X_test
    )[:, 1]

    fitted_models[model_name] = pipeline

    test_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            probabilities
        )
    })


test_results = (
    pd.DataFrame(test_rows)
    .sort_values(
        ["ROC-AUC", "F1"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(test_results.round(4))


# 9. Baseline Evaluation

Cross-validation estimates generalization across multiple folds, while the held-out test set provides a final comparison. ROC and Precision–Recall curves show discrimination across decision thresholds.


In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 6)
)

for model_name, model in fitted_models.items():

    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        name=model_name,
        ax=ax
    )

ax.set_title(
    "CreditWise — ROC Curve Comparison"
)

plt.tight_layout()
plt.show()


In [ ]:
best_model_name = cv_results.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

best_predictions = best_model.predict(
    X_test
)

print(
    "Selected model based on CV ROC-AUC:",
    best_model_name
)

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        best_predictions,
        target_names=[
            "Rejected",
            "Approved"
        ]
    )
)


In [ ]:
final_results = (
    cv_results[
        [
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC-AUC"
        ]
    ]
    .rename(
        columns={
            "Accuracy": "CV Accuracy",
            "Precision": "CV Precision",
            "Recall": "CV Recall",
            "F1": "CV F1",
            "ROC-AUC": "CV ROC-AUC"
        }
    )
)

final_results = final_results.merge(
    test_results,
    on="Model",
    how="left"
)

final_results = final_results.sort_values(
    "CV ROC-AUC",
    ascending=False
).reset_index(drop=True)

display(final_results.round(4))


# 10. XGBoost Hyperparameter Tuning

Because XGBoost was the strongest baseline, `RandomizedSearchCV` explores tree depth, learning rate, number of estimators, sampling, and regularization. Tuning is performed only on the training set with ROC-AUC as the optimization metric.


In [ ]:
if not XGBOOST_AVAILABLE:
    raise ImportError("XGBoost is required for V4.")

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / positive_count

xgb_tuning_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
        scale_pos_weight=scale_pos_weight
    ))
])

param_distributions = {
    "model__n_estimators": [150, 200, 300, 400, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.10],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_child_weight": [1, 2, 3, 5, 7],
    "model__subsample": [0.70, 0.80, 0.90, 1.00],
    "model__colsample_bytree": [0.70, 0.80, 0.90, 1.00],
    "model__gamma": [0, 0.05, 0.10, 0.20],
    "model__reg_alpha": [0, 0.001, 0.01, 0.1],
    "model__reg_lambda": [0.5, 1.0, 2.0, 5.0]
}

random_search = RandomizedSearchCV(
    estimator=xgb_tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=35,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
    return_train_score=False
)

random_search.fit(X_train, y_train)

print("Best training-CV ROC-AUC:", round(random_search.best_score_, 4))
print("\nBest parameters:")
for key, value in random_search.best_params_.items():
    print(f"{key}: {value}")

tuned_xgb = random_search.best_estimator_


In [ ]:
tuned_test_pred = tuned_xgb.predict(X_test)
tuned_test_prob = tuned_xgb.predict_proba(X_test)[:, 1]

tuned_metrics = {
    "Accuracy": accuracy_score(y_test, tuned_test_pred),
    "Precision": precision_score(y_test, tuned_test_pred, zero_division=0),
    "Recall": recall_score(y_test, tuned_test_pred, zero_division=0),
    "F1": f1_score(y_test, tuned_test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, tuned_test_prob),
    "PR-AUC": average_precision_score(y_test, tuned_test_prob),
    "Brier Score": brier_score_loss(y_test, tuned_test_prob)
}

display(
    pd.DataFrame({
        "Metric": tuned_metrics.keys(),
        "Tuned XGBoost": tuned_metrics.values()
    }).round(4)
)

print("\nClassification report:")
print(classification_report(
    y_test,
    tuned_test_pred,
    target_names=["Rejected", "Approved"]
))


In [ ]:
untuned_xgb = fitted_models["XGBoost"]
untuned_prob = untuned_xgb.predict_proba(X_test)[:, 1]
untuned_pred = untuned_xgb.predict(X_test)

tuning_comparison = pd.DataFrame([
    {
        "Model": "V3 XGBoost",
        "Accuracy": accuracy_score(y_test, untuned_pred),
        "Precision": precision_score(y_test, untuned_pred, zero_division=0),
        "Recall": recall_score(y_test, untuned_pred, zero_division=0),
        "F1": f1_score(y_test, untuned_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, untuned_prob),
        "PR-AUC": average_precision_score(y_test, untuned_prob)
    },
    {
        "Model": "Tuned XGBoost",
        "Accuracy": accuracy_score(y_test, tuned_test_pred),
        "Precision": precision_score(y_test, tuned_test_pred, zero_division=0),
        "Recall": recall_score(y_test, tuned_test_pred, zero_division=0),
        "F1": f1_score(y_test, tuned_test_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, tuned_test_prob),
        "PR-AUC": average_precision_score(y_test, tuned_test_prob)
    }
])

display(tuning_comparison.round(4))


# 11. SHAP Explainability

SHAP decomposes XGBoost predictions into feature contributions. Global importance identifies the variables that influence predictions most across applicants, while the waterfall plot explains one applicant's prediction.


In [ ]:
if SHAP_AVAILABLE:
    tuned_preprocessor = tuned_xgb.named_steps["preprocessor"]
    tuned_model = tuned_xgb.named_steps["model"]

    X_train_shap = tuned_preprocessor.transform(X_train)
    X_test_shap = tuned_preprocessor.transform(X_test)

    if hasattr(X_train_shap, "toarray"):
        X_train_shap = X_train_shap.toarray()
    if hasattr(X_test_shap, "toarray"):
        X_test_shap = X_test_shap.toarray()

    shap_feature_names = tuned_preprocessor.get_feature_names_out()

    X_train_shap_df = pd.DataFrame(
        X_train_shap,
        columns=shap_feature_names,
        index=X_train.index
    )

    X_test_shap_df = pd.DataFrame(
        X_test_shap,
        columns=shap_feature_names,
        index=X_test.index
    )

    explainer = shap.TreeExplainer(tuned_model)
    shap_values = explainer(X_test_shap_df)

    print("SHAP matrix:", shap_values.values.shape)
    print("Transformed features:", len(shap_feature_names))
else:
    print("Install SHAP and rerun this section.")


In [ ]:
if SHAP_AVAILABLE:
    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

    shap_importance = (
        pd.DataFrame({
            "Feature": shap_feature_names,
            "Mean_Absolute_SHAP": mean_abs_shap
        })
        .sort_values("Mean_Absolute_SHAP", ascending=False)
        .reset_index(drop=True)
    )

    display(shap_importance.head(20))

    shap.plots.bar(
        shap_values,
        max_display=15,
        show=True
    )


In [ ]:
if SHAP_AVAILABLE:
    shap.plots.beeswarm(
        shap_values,
        max_display=15,
        show=True
    )


In [ ]:
if SHAP_AVAILABLE:
    applicant_position = 0

    applicant_probability = tuned_xgb.predict_proba(
        X_test.iloc[[applicant_position]]
    )[0, 1]

    applicant_prediction = int(
        applicant_probability >= 0.50
    )

    print("Applicant test position:", applicant_position)
    print("Approval probability:", round(applicant_probability, 4))
    print(
        "Prediction:",
        "Approved" if applicant_prediction == 1 else "Rejected"
    )
    print(
        "Actual:",
        "Approved" if y_test.iloc[applicant_position] == 1 else "Rejected"
    )

    shap.plots.waterfall(
        shap_values[applicant_position],
        max_display=15,
        show=True
    )


# 12. Decision-Threshold Analysis

A probability threshold converts predicted approval probabilities into decisions. Instead of choosing that threshold on the test set, out-of-fold training probabilities are used to study precision, recall, and F1 across candidate thresholds.


In [ ]:
# Use the tuned hyperparameters, but obtain out-of-fold probabilities
# strictly within the training set.
threshold_model = clone(tuned_xgb)

oof_train_prob = cross_val_predict(
    threshold_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("OOF training ROC-AUC:", round(
    roc_auc_score(y_train, oof_train_prob), 4
))


In [ ]:
thresholds = np.arange(0.10, 0.91, 0.02)

threshold_rows = []

for threshold in thresholds:
    pred = (oof_train_prob >= threshold).astype(int)

    threshold_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_train, pred),
        "Precision": precision_score(y_train, pred, zero_division=0),
        "Recall": recall_score(y_train, pred, zero_division=0),
        "F1": f1_score(y_train, pred, zero_division=0)
    })

threshold_results = pd.DataFrame(threshold_rows)

best_f1_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

print("Best OOF F1 threshold:", round(best_f1_row["Threshold"], 2))
display(threshold_results.round(4))


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(threshold_results["Threshold"], threshold_results["Precision"], label="Precision")
plt.plot(threshold_results["Threshold"], threshold_results["Recall"], label="Recall")
plt.plot(threshold_results["Threshold"], threshold_results["F1"], label="F1")
plt.xlabel("Approval Probability Threshold")
plt.ylabel("Score")
plt.title("Training OOF Threshold Trade-offs")
plt.legend()
plt.tight_layout()
plt.show()


# 13. Business-Cost Analysis

Classification errors can have different consequences. This experiment assigns illustrative costs of **$5,000 for a false approval** and **$1,000 for a false rejection** to demonstrate cost-sensitive threshold selection. These values are assumptions for analysis, not real lender costs.


In [ ]:
FALSE_APPROVAL_COST = 5000
FALSE_REJECTION_COST = 1000

business_rows = []

for threshold in thresholds:
    pred = (oof_train_prob >= threshold).astype(int)

    false_approvals = int(((pred == 1) & (y_train.to_numpy() == 0)).sum())
    false_rejections = int(((pred == 0) & (y_train.to_numpy() == 1)).sum())

    total_cost = (
        false_approvals * FALSE_APPROVAL_COST
        + false_rejections * FALSE_REJECTION_COST
    )

    business_rows.append({
        "Threshold": threshold,
        "False Approvals": false_approvals,
        "False Rejections": false_rejections,
        "Estimated Cost": total_cost
    })

business_costs = pd.DataFrame(business_rows)

best_cost_row = business_costs.loc[
    business_costs["Estimated Cost"].idxmin()
]

print("Assumed false-approval cost: $", FALSE_APPROVAL_COST, sep="")
print("Assumed false-rejection cost: $", FALSE_REJECTION_COST, sep="")
print("Minimum-cost threshold:", round(best_cost_row["Threshold"], 2))
print("Estimated OOF cost: $", int(best_cost_row["Estimated Cost"]), sep="")

display(
    business_costs.sort_values("Estimated Cost").head(10)
)


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(
    business_costs["Threshold"],
    business_costs["Estimated Cost"]
)
plt.xlabel("Approval Probability Threshold")
plt.ylabel("Estimated Business Cost ($)")
plt.title("Threshold vs Estimated Business Cost")
plt.tight_layout()
plt.show()


In [ ]:
business_threshold = float(
    best_cost_row["Threshold"]
)

f1_threshold = float(
    best_f1_row["Threshold"]
)

print("Default threshold: 0.50")
print("Best training-OOF F1 threshold:", round(f1_threshold, 2))
print("Business-cost threshold:", round(business_threshold, 2))


# 14. Final Threshold Evaluation

The threshold selected from out-of-fold training predictions is applied once to the held-out test set and compared with the default 0.50 threshold. This separates model discrimination from the business decision policy.


In [ ]:
business_test_pred = (
    tuned_test_prob >= business_threshold
).astype(int)

default_test_pred = (
    tuned_test_prob >= 0.50
).astype(int)

def threshold_metrics(y_true, pred, prob, name):
    false_approvals = int(((pred == 1) & (y_true.to_numpy() == 0)).sum())
    false_rejections = int(((pred == 0) & (y_true.to_numpy() == 1)).sum())

    estimated_cost = (
        false_approvals * FALSE_APPROVAL_COST
        + false_rejections * FALSE_REJECTION_COST
    )

    return {
        "Policy": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob),
        "False Approvals": false_approvals,
        "False Rejections": false_rejections,
        "Estimated Cost": estimated_cost
    }

policy_comparison = pd.DataFrame([
    threshold_metrics(
        y_test,
        default_test_pred,
        tuned_test_prob,
        "Default 0.50"
    ),
    threshold_metrics(
        y_test,
        business_test_pred,
        tuned_test_prob,
        f"Business {business_threshold:.2f}"
    )
])

display(policy_comparison.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    business_test_pred,
    display_labels=["Rejected", "Approved"],
    ax=ax
)

ax.set_title(
    f"Tuned XGBoost — Business Threshold {business_threshold:.2f}"
)

plt.tight_layout()
plt.show()


# 15. Model Artifact

The trained pipeline and decision metadata are saved so the same preprocessing, model, and threshold policy can be reused by a deployment application.


In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

MODEL_PATH = ARTIFACT_DIR / "creditwise_xgboost_pipeline.joblib"
METADATA_PATH = ARTIFACT_DIR / "creditwise_metadata.json"

joblib.dump(tuned_xgb, MODEL_PATH)

import json as _json

metadata = {
    "model": "Tuned XGBoost",
    "target": "Loan_Approved",
    "positive_class": "Approved",
    "default_threshold": 0.50,
    "business_threshold": business_threshold,
    "false_approval_cost": FALSE_APPROVAL_COST,
    "false_rejection_cost": FALSE_REJECTION_COST,
    "random_state": RANDOM_STATE,
    "excluded_predictors": excluded_predictors,
    "engineered_features": engineered_features,
    "best_params": random_search.best_params_,
    "training_cv_roc_auc": float(random_search.best_score_),
    "test_roc_auc": float(roc_auc_score(y_test, tuned_test_prob))
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    _json.dump(metadata, f, indent=2)

print("Saved model:", MODEL_PATH)
print("Saved metadata:", METADATA_PATH)


# Conclusion

CreditWise V4 demonstrates an end-to-end credit-risk modeling workflow with leakage-safe preprocessing, cross-validated model comparison, XGBoost, SHAP explainability, and threshold-based decision analysis.

The notebook intentionally separates three questions:

- **Prediction:** How accurately can the model rank approval risk?
- **Explanation:** Which features drive the model's predictions?
- **Decision:** At what probability threshold should an approval decision be made?

Hyperparameter tuning is evaluated rather than assumed to improve performance, and business-cost values are treated as explicit analytical assumptions.
